In [ ]:
import os
import sys
from argparse import ArgumentParser, BooleanOptionalAction
import warnings
import json
import torch
import logging
import random
import numpy as np
import time
import pandas as pd
import datetime as dt
import logging.config
from tqdm import tqdm
from datasets import Dataset
# from transformers.utils import logging
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback
# from trl import DataCollatorForCompletionOnlyLM
tqdm.pandas()

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
cache_path = "/scratch/wadhwa.s/pattern_distillation/"

m = "openai-community/gpt2-xl"

In [ ]:
# model = AutoModelForCausalLM.from_pretrained(m, 
#                                                 cache_dir=cache_path, 
#                                                 device_map = "auto"
#                                                 )
tokenizer = AutoTokenizer.from_pretrained(m, 
                                            cache_dir=cache_path,
                                            # padding_side='left',
                                            use_fast=False)

In [ ]:
def prepare_preprocess_clm_fn(tokenizer: AutoTokenizer):
    def preprocess_fn(instances):
        # print (instances["clm"])
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer.pad_token = "[PAD]"
        model_inputs = tokenizer(instances["clm"], 
                                max_length=512, 
                                truncation=True, 
                                padding=True,  
                                return_tensors="pt",
                                ).to(device)
        return model_inputs

    return preprocess_fn

In [ ]:
data_path = "/work/frink/shaib.c/pattern_distillation/generated_data/"
alpaca_llama = os.path.join(data_path, "alpaca_generated_Meta-Llama-3.1-70B-Instruct-Turbo.jsonl")

In [ ]:
data = alpaca_llama

with open(data, 'r') as f:
    data = [json.loads(line) for line in f]


In [ ]:
print ("Len of data: ", len(data))

In [ ]:
df = pd.DataFrame(data)
df = df.astype(str) 
print (df.shape)

df.head()

In [ ]:
for ix, row in df.sample(50).iterrows():
    print ("text: ", row["text"])
    print ("\ngenerated_summary: ", row["generated_summary"])
    print ("\n-------------------\n")

In [ ]:
test_path = os.path.join(data_path, "alpaca_generated_Meta-Llama-3.1-70B-Instruct-Turbo_test.csv")

In [ ]:
d = Dataset.from_pandas(df)
d = d.train_test_split(test_size=0.2)
d["test"].to_csv(test_path, index=False)
df_test = pd.read_csv(test_path)
df_test.head()

In [ ]:
df["clm"] = tokenizer.bos_token + df["text"] + "#### [SUMMARY]" + df["generated_summary"] + "[SUMMARY]"
df.head()

# df["clm"] = tokenizer.bos_token + df["article"]# + "#### [SUMMARY]" + df["abstract"] + "[SUMMARY]"
# df.head()

In [ ]:
df["id"] = pd.to_numeric(df["id"])

# df_new = df[df["id"].isin(common_ids)]
# print (df_new.shape)
# df = df_new

In [ ]:
df.shape, df_test.shape

In [ ]:
df['token_length'] = df['clm'].progress_apply(lambda x: len(tokenizer.tokenize(x)))
df = df.sort_values(by='token_length', ascending=True)
df_filtered = df.sample(frac=1, random_state=42)

In [ ]:
# df_filtered = df.iloc[1250:3000].sample(frac=1, random_state=42) # sft/training
# df_filtered = df.iloc[650:1250].sample(frac=1, random_state=42) # inference/held out test
mean_token_length = df_filtered['token_length'].mean()
median_token_length = df_filtered['token_length'].median()
max_token_length = df_filtered['token_length'].max()
print("Mean token length:", mean_token_length)
print("Median token length:", median_token_length)
print("Max token length:", max_token_length)

In [ ]:
mean_token_length = df['token_length'].mean()
median_token_length = df['token_length'].median()
print("Mean token length:", mean_token_length)
print("Median token length:", median_token_length)

In [ ]:
train_path = os.path.join(data_path, "alpaca_generated_Meta-Llama-3.1-70B-Instruct-Turbo_filtered.csv")

In [ ]:
df_filtered.to_csv(train_path, index=False)
df_filtered.shape

In [ ]:
df = pd.read_csv(train_path)
df.head()

In [ ]:
d = Dataset.from_pandas(df)
d = d.train_test_split(test_size=0.2)
d_train = d['train']
d_valid = d['test']

In [ ]:
d_train

In [ ]:
preprocess_fn = prepare_preprocess_clm_fn(tokenizer)

In [ ]:
tokenized_d_train = d_train.map(preprocess_fn, batched=True)
tokenized_d_valid = d_valid.map(preprocess_fn, batched=True)

In [ ]:
tokenized_d_train = tokenized_d_train.remove_columns(d_train.column_names)
tokenized_d_valid = tokenized_d_valid.remove_columns(d_valid.column_names)

In [ ]:
print (d_train[0]["clm"])

In [ ]:
output_path = cache_path + "pattern_distill_models/"

In [ ]:
torch.cuda.device_count()

In [ ]:
response_template = "#### [SUMMARY]"

In [ ]:
data_collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer, mlm=False)

In [ ]:
training_args = TrainingArguments(
                # report_to=args.report_to,
                output_dir=output_path,
                evaluation_strategy="steps",    
                eval_steps=500,
                learning_rate=3e-4,
                save_strategy="steps",
                per_device_train_batch_size=10,
                per_device_eval_batch_size=10,
                auto_find_batch_size=False,
                eval_delay=200,
                logging_strategy="steps",
                logging_steps=1000,
                weight_decay=0.01,
                save_total_limit=3,
                num_train_epochs=30,
                logging_dir=output_path + "/logs",
                load_best_model_at_end = True,
                metric_for_best_model = "eval_loss",
                greater_is_better = False,
                # log_level="info",
                eval_accumulation_steps=50,
                gradient_accumulation_steps=4
                )

In [ ]:
trainer = Trainer(
                model=model,
                args=training_args,
                data_collator=data_collator,
                train_dataset=tokenized_d_train,
                eval_dataset=tokenized_d_valid,
                tokenizer=tokenizer,
                # compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.02)]
                )

In [ ]:
trainer.train()

In [ ]:
df1 = pd.read_csv("/work/frink/shaib.c/pattern_distillation/generated_data/pubmedsum_generated_Meta-Llama-3.1-70B-Instruct-Turbo_test.csv")
df2 = pd.read_csv("/work/frink/shaib.c/pattern_distillation/generated_data/pubmedsum_generated_Meta-Llama-3.1-8B-Instruct_test.csv")
df3 = pd.read_csv("/work/frink/shaib.c/pattern_distillation/generated_data/pubmedsum_generated_Mistral-7B-Instruct-v0.3_test.csv")

In [ ]:
df = df1.merge(df2, on='id').merge(df3, on='id')

# df = df1.merge(df2, on='id')

In [ ]:
common_ids = df["id"].tolist()
print (len(common_ids))

In [ ]:
df1 = df1[df1.id.isin(common_ids)]
df2 = df2[df2.id.isin(common_ids)]
df3 = df3[df3.id.isin(common_ids)]

In [ ]:
# df3.to_csv("/work/frink/shaib.c/pattern_distillation/generated_data/cnn_dailymail_generated_Mistral-7B-Instruct-v0.3_test.csv", index=False)

In [ ]:
df = pd.read_csv("/work/frink/shaib.c/pattern_distillation/original_data/pubmed_summ.csv")

In [ ]:
df1.to_csv("/work/frink/shaib.c/pattern_distillation/generated_data/pubmedsum_generated_Meta-Llama-3.1-70B-Instruct-Turbo_test.csv", index=False)

In [ ]:
df2.to_csv("/work/frink/shaib.c/pattern_distillation/generated_data/pubmedsum_generated_Meta-Llama-3.1-8B-Instruct_test.csv", index=False)

In [ ]:
df3.to_csv("/work/frink/shaib.c/pattern_distillation/generated_data/pubmedsum_generated_Mistral-7B-Instruct-v0.3_test.csv", index=False)

In [ ]:
df4 = pd.read_csv("/work/frink/shaib.c/pattern_distillation/generated_data/pubmedsum_generated_Meta-Llama-3.1-8B-Instruct_test.csv")

In [ ]:
d = Dataset.from_pandas(df4)

In [ ]:
d = d.train_test_split(test_size=0.2)
d

In [ ]:
# replace d["test"] with only first 50 instances of d["test"]
d["test"] = d["test"].select(range(50))

In [ ]:
d

In [ ]:
df = pd.read_csv(alpaca_llama)
df.head()

In [ ]:
df.drop(columns=["token_length", "__index_level_0__"], inplace=True)

In [ ]:
df.to_csv(test_path, index=False)

In [ ]:
df.shape